<a href="https://colab.research.google.com/github/SelinBeyaz01/BUS392-Steam-Project/blob/Anjani-Analysis/BUS392_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install kagglehub

In [7]:
import kagglehub
path = kagglehub.dataset_download("hubertsidorowicz/steam-games-dataset-daily-updates")

Using Colab cache for faster access to the 'steam-games-dataset-daily-updates' dataset.


In [8]:
import pandas as pd
gamesData_location = path + "/steam_games.csv"
gamesData = pd.read_csv(gamesData_location)
gamesData.head()

/tmp/ipykernel_1685/1104197516.py:3: DtypeWarning: Columns (30,31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  gamesData = pd.read_csv(gamesData_location)


,app_id,name,release_date,price,price_status,estimated_owners,developers,publishers,genres,categories,...,linux,header_image,screenshots,movies,website,support_url,support_email,metacritic_url,steam_store_available,steam_spy_available
0,2582880,Northrealm Tribes,2026-10-20,0.00,free,"0 .. 20,000","[""DevSanja""]","[""DevSanja""]","[{""id"": ""4"", ""description"": ""Casual""}, {""id"": ...","[{""id"": 2, ""description"": ""Single-player""}, {""...",...,True,https://shared.akamai.steamstatic.com/store_it...,"[{""id"": 0, ""path_thumbnail"": ""https://shared.a...","[{""id"": 257460676, ""name"": ""Northrealm Tribes:...",NaN,NaN,ma.schleufe@gmx.de,NaN,True,True
1,3508520,Echoes of the Abyss,2026-10-21,0.99,paid,"0 .. 20,000","[""lijj1235666""]","[""lij1235666""]","[{""id"": ""1"", ""description"": ""Action""}, {""id"": ...","[{""id"": 2, ""description"": ""Single-player""}, {""...",...,False,https://shared.akamai.steamstatic.com/store_it...,"[{""id"": 0, ""path_thumbnail"": ""https://shared.a...","[{""id"": 257108877, ""name"": ""Bounce ball simula...",NaN,NaN,q1w2e3_r42023@qq.com,NaN,True,True
2,1508190,Finding Words,2026-10-21,3.99,paid,"0 .. 20,000","[""Odd Build""]","[""Odd Build""]","[{""id"": ""4"", ""description"": ""Casual""}]","[{""id"": 2, ""description"": ""Single-player""}, {""...",...,False,https://shared.akamai.steamstatic.com/store_it...,"[{""id"": 0, ""path_thumbnail"": ""https://shared.a...","[{""id"": 257459902, ""name"": ""Finding Words Trai...",NaN,criandomeujogo.com,criandomeujogo@hotmail.com,NaN,True,True
3,3639110,Terror In The Shadowrealms,2026-10-21,1.99,paid,"0 .. 20,000","[""Onigumo69""]","[""Onigumo69""]","[{""id"": ""4"", ""description"": ""Casual""}, {""id"": ...","[{""id"": 2, ""description"": ""Single-player""}, {""...",...,False,https://shared.akamai.steamstatic.com/store_it...,"[{""id"": 0, ""path_thumbnail"": ""https://shared.a...","[{""id"": 257421609, ""name"": ""Terror In The Shad...",NaN,NaN,onigumo69@outlook.com,NaN,True,True
4,1564060,Lighthouse of Madness,2026-10-21,13.99,paid,"0 .. 20,000","[""Flying Jack Studios""]","[""Flying Jack Productions""]","[{""id"": ""25"", ""description"": ""Adventure""}, {""i...","[{""id"": 2, ""description"": ""Single-player""}, {""...",...,False,https://shared.akamai.steamstatic.com/store_it...,"[{""id"": 0, ""path_thumbnail"": ""https://shared.a...","[{""id"": 256900399, ""name"": ""Lighthouse of Madn...",https://lighthouseofmadness.com,https://www.flyingjack.com.br,productions@flyingjack.com.br,NaN,True,True


In [11]:
import pandas as pd
import numpy as np

analysis = gamesData[['app_id', 'name', 'release_date', 'estimated_owners', 'positive', 'negative']].copy()
analysis['release_date'] = pd.to_datetime(analysis['release_date'], errors='coerce')
analysis['positive'] = pd.to_numeric(analysis['positive'], errors='coerce').fillna(0)
analysis['negative'] = pd.to_numeric(analysis['negative'], errors='coerce').fillna(0)
analysis['total_reviews'] = analysis['positive'] + analysis['negative']
analysis = analysis.dropna(subset=['app_id', 'name', 'release_date'])
analysis = analysis.drop_duplicates(subset='app_id')

def owner_midpoint_fixed(owner_range):
    try:
        clean_range = str(owner_range).replace(',', '').replace(' - ', ' .. ')
        low, high = clean_range.split(' .. ')
        return (int(low) + int(high)) / 2
    except:
        return np.nan

analysis['estimated_owners_num'] = analysis['estimated_owners'].apply(owner_midpoint_fixed)

reference_date = analysis['release_date'].max()
analysis['age_days'] = (reference_date - analysis['release_date']).dt.days
analysis['age_years'] = analysis['age_days'] / 365.25
analysis['age_months'] = analysis['age_days'] / 30.44

age_data = analysis[(analysis['age_months'] >= 6) & (analysis['total_reviews'] > 0)].copy()

age_data['positive_per_year'] = age_data['positive'] / age_data['age_years']
age_data['negative_per_year'] = age_data['negative'] / age_data['age_years']
age_data['positive_per_month'] = age_data['positive'] / age_data['age_months']
age_data['negative_per_month'] = age_data['negative'] / age_data['age_months']

def age_group(years):
    if years < 2:
        return '1) Under 2 years'
    elif years < 5:
        return '2) 2-5 years'
    elif years < 10:
        return '3) 5-10 years'
    else:
        return '4) 10+ years'

age_data['age_group'] = age_data['age_years'].apply(age_group)

print("Reference date:", reference_date.date())
print("Games used in the age analysis:", len(age_data))
age_data['age_group'].value_counts().sort_index()

Reference date: 2026-11-22
Games used in the age analysis: 83028


,count
age_group,
1) Under 2 years,7455
2) 2-5 years,30128
3) 5-10 years,35911
4) 10+ years,9534
